# Indirect-noise (entropy-driven) thermoacoustic instability

A compact flame in a duct that ends in a **choked nozzle** can go unstable through a path
that pure acoustics cannot see.  The unsteady heat release sheds an **entropy spot** (a hot/cold
lump) as well as a sound wave; the spot **convects** down the hot duct and, at the accelerating
choked nozzle, is partly converted **back into sound** (Marble--Candel "indirect" combustion
noise).  That reflected wave travels upstream to the flame and closes a feedback loop.  If the
loop gain and phase are right, a mode grows -- an instability that exists *only because the
convected entropy wave is retained*.

This notebook uses the **Nyquist open-loop stability driver**
(:func:`fns.perturbation.open_loop_response`).  It evaluates the flame's *return ratio* on the
**real frequency axis**, so the convected entropy wave is handled exactly (bounded phase
$|e^{-i\omega\tau}|=1$), and it counts the encirclements of the critical point to get the number
of unstable modes.  We will see:

* with the entropy wave **dropped** (`isentropic=True`, the textbook acoustic assumption) the rig
  is **stable**;
* with it **retained** (`isentropic=False`) one mode is **unstable** -- the indirect-noise loop;
* the eigenvalue solver (:func:`fns.perturbation.eigenmodes`) agrees on the count and pins the
  growing mode, validating the real-axis driver.

In [ ]:
import os, sys, warnings

_root = os.getcwd()
while not os.path.isdir(os.path.join(_root, "fns")) and _root != os.path.dirname(_root):
    _root = os.path.dirname(_root)
sys.path.insert(0, _root)

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from fns.elements import catalog as cat
from fns.elements.dynamic_source import heat_release_response, n_tau_lowpass, tabulated
from fns.perturbation.operator.boundary_bc import PerturbationBC
from fns.perturbation import open_loop_response, eigenmodes, forced_response
from fns.solver import solve
from fns.solver.control import states_table
from fns.thermo.configure import perfect_gas
from fns.assembly.derive import ES_T, ES_M, ES_U, ES_C, ES_RHO
from fns.plotting import use_fns_theme

use_fns_theme()
R_AIR, GAMMA = 287.0, 1.4
CP = GAMMA * R_AIR / (GAMMA - 1.0)

## 1. The rig

Cold air enters through a **mass-flow inlet** (its *inherited* acoustic closure -- a passive,
constant-mass-flow termination).  A compact heat-release flame carries a flame transfer function
(FTF): a low-pass `n-tau` model (a real flame rolls its gain off above a cutoff, which a bare
`n-tau` does not).  The hot duct ends in a compact **choked nozzle**, the element that converts
the convected entropy spot back into sound.

The flame + choked nozzle will not converge from a cold guess, so we ramp the heat release
(continuation), reusing the previous solution.

In [ ]:
AREA, A_STAR, MDOT = 0.02, 0.012, 0.4      # duct area, nozzle throat area, mass flow [kg/s]
L1, L2 = 0.5, 0.8                          # cold (approach) and hot (flame->nozzle) duct lengths [m]
DT = 750.0                                 # target mean temperature rise across the flame [K]
N_FTF, TAU, FC = 0.6, 3.0e-3, 300.0        # FTF gain, time lag [s], roll-off cutoff [Hz]

ftf = heat_release_response(n_tau_lowpass(N_FTF, TAU, FC), ref_edge=1)  # heat release responds to u' on edge 1

def build(Qdot):
    els = [
        cat.mass_flow_inlet(MDOT, 300.0, perturbation_bc=PerturbationBC.inherit()),
        cat.duct(L1),
        cat.heat_release_flame(Qdot, dynamic_source=ftf),
        cat.duct(L2),
        cat.choked_nozzle_outlet(A_STAR),
    ]
    edges = [(0, 1, AREA), (1, 2, AREA), (2, 3, AREA), (3, 4, AREA)]
    return cat.build_problem(perfect_gas(R_AIR, GAMMA), els, edges, mdot_ref=MDOT, p_ref=1e5, h_ref=CP * 300.0)

x = None
for ddT in (1.0, 0.25 * DT, 0.5 * DT, 0.75 * DT, DT):   # heat-release continuation
    prob = build(MDOT * CP * ddT)
    res = solve(prob, x0=x)
    assert res.converged
    x = res.x

est = states_table(prob, x)
u_hot, c_hot = float(est[ES_U, 2]), float(est[ES_C, 2])
print(f"hot gas: T = {est[ES_T, 2]:.0f} K,  u = {u_hot:.0f} m/s,  M = {est[ES_M, 2]:.3f}")
print(f"nozzle inlet Mach = {est[ES_M, 3]:.3f}")
print(f"entropy convective lag flame->nozzle  L2/u = {L2 / u_hot * 1e3:.2f} ms")

## 2. Pure-acoustic stability: drop the entropy wave

Set `isentropic=True` -- the standard acoustic assumption pins the entropy wave to zero in the
ducts ($\rho' = p'/c^2$).  The flame still drives sound, but the convected entropy spot cannot
reach the nozzle, so the indirect-noise path is severed.  The Nyquist return ratio
$L(\omega)$ does **not** encircle the critical point $+1$: **stable**.

In [ ]:
freqs = np.linspace(0.0, 1600.0, 1200)
nyq_iso = open_loop_response(prob, x, freqs, isentropic=True)
print("isentropic (acoustic only):  n_unstable =", nyq_iso.n_unstable,
      " margin =", round(nyq_iso.margin, 3), " closed =", nyq_iso.closed)

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    eig_iso = eigenmodes(prob, x, freq_band=(60.0, 900.0), growth_band=(-260.0, 120.0),
                         isentropic=True, residual_tol=1e-8)
g_iso = [(round(f, 1), round(g, 1)) for f, g in zip(eig_iso.freqs, eig_iso.growth_rates)]
print("eigenmodes (acoustic only): modes (f Hz, growth 1/s) =", g_iso)
print("  unstable modes =", int(np.sum(eig_iso.unstable)),
      "-> all decay" if not np.any(eig_iso.unstable) else "-> UNSTABLE")

## 3. Retain the entropy wave: the indirect-noise instability

Now `isentropic=False` keeps the convected entropy wave.  The Nyquist locus encircles $+1$
**once** -- one unstable mode -- and the eigenvalue solver, run on the *same* operator, finds a
**growing** mode at the same count.  The instability appeared purely from re-including the
entropy/indirect-noise path.

In [ ]:
nyq_full = open_loop_response(prob, x, freqs, isentropic=False)
print("full (entropy retained):  n_unstable =", nyq_full.n_unstable,
      " margin =", round(nyq_full.margin, 3), " closed =", nyq_full.closed)

with warnings.catch_warnings():
    warnings.simplefilter("ignore")   # the convective spectrum is dense; we read the growing mode
    eig_full = eigenmodes(prob, x, freq_band=(60.0, 200.0), growth_band=(-100.0, 120.0),
                          isentropic=False, residual_tol=1e-8)
unstable = [(f, g) for f, g in zip(eig_full.freqs, eig_full.growth_rates) if g > 0]
print("eigenmodes (entropy retained): growing mode(s) =",
      [(round(f, 1), round(g, 1)) for f, g in unstable], "  (freq Hz, growth 1/s)")

**The acoustic analysis calls the rig stable; retaining the entropy wave makes it unstable.**
The Nyquist count (encirclements of $+1$) and the eigensolver (a genuine growing eigenmode) agree
-- the entropy/acoustic coupling is real and shows up in the stability analysis.

## 4. The Nyquist diagram

The return ratio $L(\omega) = -F(\omega)\,b^{\mathsf T}A_0^{-1}a$ is the flame's open-loop gain
(the FTF $F$ times the acoustic+entropy "plant" $b^{\mathsf T}A_0^{-1}a$).  An instability is an
**encirclement of the critical point $+1$**.  Dropping the entropy wave pulls the locus off the
critical point; keeping it wraps the locus around $+1$.

In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=(
    "acoustic only (isentropic) -- stable", "entropy retained (full) -- unstable"))
for col, r in ((1, nyq_iso), (2, nyq_full)):
    fig.add_scatter(x=r.L.real, y=r.L.imag, mode="lines", name="L(w), w>0",
                    line=dict(color="#1f77b4"), showlegend=(col == 1), row=1, col=col)
    fig.add_scatter(x=r.L.real, y=-r.L.imag, mode="lines", name="L(w), w<0 (mirror)",
                    line=dict(color="#1f77b4", dash="dot"), showlegend=(col == 1), row=1, col=col)
    fig.add_scatter(x=[1.0], y=[0.0], mode="markers", marker=dict(symbol="x", size=12, color="#d62728"),
                    name="critical +1", showlegend=(col == 1), row=1, col=col)
fig.update_xaxes(title_text="Re L", row=1, col=1); fig.update_xaxes(title_text="Re L", row=1, col=2)
fig.update_yaxes(title_text="Im L", row=1, col=1)
fig.update_layout(title="Nyquist locus of the flame return ratio L(w)")
fig.show()

In [ ]:
# distance to the critical point, |det(I - L)| = |1 - L|, vs frequency
fig = go.Figure()
fig.add_scatter(x=nyq_iso.freqs, y=np.abs(nyq_iso.D), mode="lines", name="acoustic only (stable)")
fig.add_scatter(x=nyq_full.freqs, y=np.abs(nyq_full.D), mode="lines", name="entropy retained (unstable)")
fig.add_hline(y=0.0, line=dict(color="#d62728", dash="dash"))
fig.update_layout(title="Stability margin |1 - L(f)| -- dips mark the least-stable frequencies",
                  xaxis_title="frequency [Hz]", yaxis_title="|1 - L|")
fig.show()

## 5. The mechanism, seen in a forced response

Drive the inlet with a unit acoustic wave and read the waves around the loop (real frequencies,
entropy retained).  The flame turns the forcing into an **entropy wave** $h$ that is absent
upstream, convects to the nozzle conserving its magnitude, and there splits the reflected sound
into a **direct** part $R f$ and an **indirect** (entropy) part $R_s h$ -- the feedback that drove
the instability.

In [ ]:
# rebuild with an inlet excitation to probe the loop
els_exc = [
    cat.mass_flow_inlet(MDOT, 300.0, perturbation_bc=PerturbationBC.anechoic(driven=("acoustic",))),
    cat.duct(L1),
    cat.heat_release_flame(MDOT * CP * DT, dynamic_source=ftf),
    cat.duct(L2),
    cat.choked_nozzle_outlet(A_STAR),
]
prob_exc = cat.build_problem(perfect_gas(R_AIR, GAMMA), els_exc,
                             [(0, 1, AREA), (1, 2, AREA), (2, 3, AREA), (3, 4, AREA)],
                             mdot_ref=MDOT, p_ref=1e5, h_ref=CP * 300.0)
fsw = np.linspace(60.0, 900.0, 240)
fr = forced_response(prob_exc, x, fsw, isentropic=False)
h_cold = np.abs(fr.waves(1)[:, 2])    # entropy on the cold approach (should be ~0)
h_hot = np.abs(fr.waves(2)[:, 2])     # entropy just downstream of the flame
h_noz = np.abs(fr.waves(3)[:, 2])     # entropy arriving at the nozzle

# Marble-Candel split at the nozzle: g = R f + R_s h
rho, c, M = float(est[ES_RHO, 3]), float(est[ES_C, 3]), float(est[ES_M, 3])
R = (2.0 - (GAMMA - 1.0) * M) / (2.0 + (GAMMA - 1.0) * M)
R_s = (c / rho) * M / (2.0 + (GAMMA - 1.0) * M)
f3, g3, h3 = (fr.waves(3)[:, k] for k in (0, 1, 2))
direct, indirect = np.abs(R * f3), np.abs(R_s * h3)

fig = make_subplots(rows=1, cols=2, subplot_titles=(
    "entropy wave |h| around the loop", "reflected sound at the nozzle: direct vs indirect"))
fig.add_scatter(x=fsw, y=h_cold, name="cold approach", row=1, col=1)
fig.add_scatter(x=fsw, y=h_hot, name="flame outlet", row=1, col=1)
fig.add_scatter(x=fsw, y=h_noz, name="nozzle inlet", row=1, col=1, line=dict(dash="dot"))
fig.add_scatter(x=fsw, y=direct, name="direct |R f|", row=1, col=2)
fig.add_scatter(x=fsw, y=indirect, name="indirect |R_s h|", row=1, col=2)
fig.update_xaxes(title_text="frequency [Hz]", row=1, col=1); fig.update_xaxes(title_text="frequency [Hz]", row=1, col=2)
fig.update_layout(title="Indirect combustion noise: the flame's entropy spot converts to sound at the nozzle")
fig.show()
print("entropy upstream of the flame is negligible: max |h_cold| =", f"{h_cold.max():.2e}")

## 6. Why the real-axis driver: a measured (tabulated) flame response

A real FTF is **measured**, returned as a table of complex gain vs frequency.  The eigenvalue
solver searches *complex* frequencies, so it cannot continue a real-grid table and **refuses**
it.  The Nyquist driver evaluates the FTF on the real axis, so a tabulated flame response works
directly -- and gives the same verdict as the closed-form model.

In [ ]:
f_tab = np.linspace(0.0, 2500.0, 300)
F_tab = np.array([n_tau_lowpass(N_FTF, TAU, FC)(f) for f in f_tab])   # sample the model as "measured" data
ftf_meas = heat_release_response(tabulated(f_tab, F_tab), ref_edge=1)
els_meas = [
    cat.mass_flow_inlet(MDOT, 300.0, perturbation_bc=PerturbationBC.inherit()),
    cat.duct(L1), cat.heat_release_flame(MDOT * CP * DT, dynamic_source=ftf_meas),
    cat.duct(L2), cat.choked_nozzle_outlet(A_STAR),
]
prob_meas = cat.build_problem(perfect_gas(R_AIR, GAMMA), els_meas,
                              [(0, 1, AREA), (1, 2, AREA), (2, 3, AREA), (3, 4, AREA)],
                              mdot_ref=MDOT, p_ref=1e5, h_ref=CP * 300.0)
nyq_meas = open_loop_response(prob_meas, x, freqs, isentropic=False)
print("tabulated (measured) FTF via Nyquist:  n_unstable =", nyq_meas.n_unstable)
try:
    eigenmodes(prob_meas, x, freq_band=(60.0, 200.0), isentropic=False)
except ValueError as e:
    print("eigenmodes with a tabulated FTF ->", type(e).__name__, ":", str(e).split('.')[0])

## Summary

* A flame feeding a **choked nozzle** can be **stable as a pure acoustic system yet unstable once
  the convected entropy wave is retained** -- the indirect-noise feedback (entropy spot $\to$
  nozzle $\to$ sound) is the destabilizer.
* The **Nyquist open-loop driver** resolves it on the real frequency axis: the flame return ratio
  encircles the critical point $+1$ once (`n_unstable = 1`), matching the growing eigenmode the
  eigenvalue solver finds (~120 Hz) -- the entropy/acoustic coupling shows up in the stability
  analysis.
* The real-axis driver also accepts a **measured / tabulated FTF**, which the complex-plane
  eigensolver cannot -- the practical route to stability assessment with experimental flame data.